In [ ]:
# NOTE: use FastLloyd/env.yml
import sys, os
from pathlib import Path

def _find_project_root():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError("Could not find project root (.git not found)")

root = str(_find_project_root())
os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

In [2]:
from src.helpers import *
import numpy as np

datasets, k_dict, r_dict, l_inf_dict = get_grid_datasets_final()

In [3]:
# File paths — our methods
PE_FOLDER     = 'result_final/ours_final_may14'
HDPE_FOLDER   = 'result_final/ours_dim_reduced_may14'

# File paths — baselines
GOOGLE_FOLDER      = 'result_final/google_baseline_50'
FASTLLOYD_FOLDER   = 'result_final/fastlloyd_baseline_50'
DIFFPRIVLIB_FOLDER = 'result_final/diffprivlib_baseline_50'
ICML_FOLDER        = 'result_final/icml_baseline_50_jun24'

In [4]:
import pandas as pd
import numpy as np
import pickle
from collections import defaultdict

rows = []

# Helper to handle the NumPy 1.x vs 2.x name change
def calculate_auc(y, x):
    if hasattr(np, 'trapezoid'):
        return np.trapezoid(y, x=x)
    return np.trapz(y, x=x)

def get_mean_loss(path, dataset_size, eps):
    try:
        with open(path, 'rb') as f:
            data = pickle.load(f)
            return np.mean(data['loss'][str(eps)]) / dataset_size
    except:
        return np.nan

# 1. Collect Data
for dataset_name, dataset in datasets.items():
    result = {'dataset': dataset_name, 'N': dataset.shape[0],'d': dataset.shape[1], 'k': k_dict[dataset_name]}
    size = dataset.shape[0]

    # --- Process PE (Our Method) ---
    try:
        with open(f'{PE_FOLDER}/grid_results_{dataset_name}.pkl', 'rb') as f:
            list_tuples = pickle.load(f)
        
        config_convergences = defaultdict(list)
        for config, loss, centers, convergence in list_tuples:
            config_convergences[config['eps']].append(convergence[-1])
        
        pe_curve = []
        for e in EPSILONS:
            val = np.mean(config_convergences[e]) / size if e in config_convergences else np.nan
            pe_curve.append(val)
        
        result['PE'] = calculate_auc(pe_curve, EPSILONS) if not np.isnan(pe_curve).any() else np.nan
    except:
        result['PE'] = np.nan

    # Process our method (PE+)
    try:
        with open(f'{HDPE_FOLDER}/grid_results_{dataset_name}.pkl', 'rb') as f:
            list_tuples = pickle.load(f)
        config_convergences_n = defaultdict(list)
        for config, loss, centers, convergence in list_tuples:
            config_convergences_n[config['eps']].append(loss)
        
        pe_n_curve = []
        for e in EPSILONS:
            val = np.mean(config_convergences_n[e]) / size 
            pe_n_curve.append(val)
        
        result['Projected PE'] = calculate_auc(pe_n_curve, EPSILONS) if dataset.shape[1] > 32  else np.nan
    except Exception as e:
        print(e)
        result['Projected PE'] = np.nan

    # --- Process Baselines ---
    google_path = f'{GOOGLE_FOLDER}/baseline_{dataset_name}.pkl'
    # Treat non-private loss as constant across epsilon and integrate — same scale as private AUC columns
    non_private_loss = get_mean_loss(google_path, size, eps='inf')
    result['non_private'] = calculate_auc([non_private_loss] * len(EPSILONS), EPSILONS)
    
    baseline_files = {
        'google':      google_path,
        'fastlloyd':   f'{FASTLLOYD_FOLDER}/fastlloyd_{dataset_name}.pkl',
        'diffprivlib': f'{DIFFPRIVLIB_FOLDER}/diffpriv_{dataset_name}.pkl',
        'icml':        f'{ICML_FOLDER}/baseline_{dataset_name}.pkl',
    }

    for algo, path in baseline_files.items():
        curve = [get_mean_loss(path, size, e) for e in EPSILONS]
        result[algo] = calculate_auc(curve, EPSILONS) if not np.isnan(curve).any() else np.nan
    
    rows.append(result)

# 2. Create and Style DataFrame
df = pd.DataFrame(rows)
cols = ['dataset', 'N', 'd', 'k', 'non_private', 'PE', 'Projected PE', 'google', 'fastlloyd', 'diffprivlib', 'icml']
df = df[cols]

def highlight_winners(s):
    numeric_s = pd.to_numeric(s, errors='coerce').dropna()
    if numeric_s.empty: return [''] * len(s)
    return ['font-weight: bold; background-color: #d4edda' if v == numeric_s.min() else '' for v in s]

private_algos = ['PE', 'Projected PE', 'google', 'fastlloyd', 'diffprivlib', 'icml']

styled_df = df.style.apply(highlight_winners, axis=1, subset=private_algos) \
                   .format(precision=4, na_rep="-") \
                   .set_caption(f"AUC of Loss across epsilon values")

styled_df

,dataset,N,d,k,non_private,PE,Projected PE,google,fastlloyd,diffprivlib,icml
0,birch2,25000,2,100,0.0001,0.0003,-,0.0658,0.0031,0.0124,0.0161
1,iris,150,4,3,0.1361,0.2894,-,0.6892,0.3979,1.3048,5.5766
2,adult,48842,6,3,0.0056,0.0056,-,0.0076,0.0155,0.0213,0.0113
3,mnist,70000,84,10,0.1958,0.2920,0.2169,0.2173,0.5822,0.5315,0.7904
4,letter,20000,16,26,0.2481,0.2852,-,0.3187,0.3963,0.4341,0.6277
5,gas,36733,12,6,0.1026,0.1081,-,0.1121,0.1561,0.1761,0.1853
6,g2_4,2048,4,2,0.4650,0.4761,-,0.5465,0.4666,0.5468,0.8802
7,g2_16,2048,16,2,0.7677,0.7800,-,0.8391,0.8687,1.1816,4.7489
8,g2_64,2048,64,2,1.0956,1.1538,1.1295,1.2478,1.9647,2.6608,27.5361
9,g2_128,2048,128,2,1.3534,1.7237,1.4055,1.5906,2.6425,4.4632,74.0978


In [5]:
# 2. Create and Strictly Order the DataFrame
df = pd.DataFrame(rows)

# --- Compute Relative Improvement Column ---
private_algos_raw = ['PE', 'Projected PE', 'google', 'fastlloyd', 'diffprivlib', 'icml']
our_methods_raw = ['PE', 'Projected PE']

def compute_net_improvement(row):
    vals = row[private_algos_raw].dropna().astype(float)
    our_vals = vals[vals.index.isin(our_methods_raw)].dropna()
    
    if vals.empty or our_vals.empty:
        return np.nan
        
    actual_winner = vals.idxmin()
    best_our_loss = our_vals.min()
    
    # Best competitor = best among related work only (never one of our methods)
    competitor_vals = vals[~vals.index.isin(our_methods_raw)]
    if competitor_vals.empty:
        return 0.0
    best_competitor_loss = competitor_vals.min()
    
    if actual_winner in our_methods_raw:
        # We won: % by which our best is lower than best competitor
        return ((best_competitor_loss - best_our_loss) / best_competitor_loss) * 100
    else:
        # We lost: % by which our best is higher than best competitor (negative = worse)
        return ((best_competitor_loss - best_our_loss) / best_competitor_loss) * 100

df['Our Improvement'] = df.apply(compute_net_improvement, axis=1)

# Define the exact order of internal keys including the new column
original_cols = ['dataset', 'N', 'd', 'k', 'non_private', 'PE', 'Projected PE', 'google', 'fastlloyd', 'diffprivlib', 'icml', 'Our Improvement']
df = df[original_cols]

# Use an explicit map to prevent column-swapping errors
rename_map = {
    'dataset': 'Dataset',
    'N' : '$N$',
    'd': '$d$',
    'k': '$k$',
    'non_private': 'Non-Priv',
    'PE': 'PE',
    'Projected PE' : 'HDPE',
    'google': 'Google',
    'fastlloyd': 'FastLloyd',
    'diffprivlib': 'DP-Lib',
    'icml': 'ICML17',
    'Our Improvement': 'Improv.'
}
df = df.rename(columns=rename_map)

# Replace underscores in Dataset names for LaTeX compatibility
df['Dataset'] = df['Dataset'].str.replace('synthetic_', '', regex=False)
df['Dataset'] = df['Dataset'].str.replace('_', r'\_', regex=False)


# Define which columns participate in the highlighting (the private algorithms)
private_algos = ['PE', 'HDPE', 'Google', 'FastLloyd', 'DP-Lib', 'ICML17']

def highlight_winners(s):
    numeric_s = pd.to_numeric(s, errors='coerce').dropna()
    if numeric_s.empty: return [''] * len(s)
    return ['font-weight: bold; background-color: #d4edda' if v == numeric_s.min() else '' for v in s]

# 3. Apply Styling and Export
format_dict = {col: "{:.4f}" for col in private_algos + ['Non-Priv']}
# Added the backslash escape (\%) to the string format
format_dict['Improv.'] = r"{:+.2f}\%"

styled_df = df.style.apply(highlight_winners, axis=1, subset=private_algos) \
                   .format(format_dict, na_rep="-") \
                   .hide(axis="index")

# Export to LaTeX
latex_code = styled_df.to_latex(
    column_format='lrrr|r|rrrrrr|r', # Fixed 'ffffff' to 'rrrrrr' 
    hrules=True,
    convert_css=True,
    caption=r"AUC of Loss across $\epsilon$ values with Relative Improvements",
    label="tab:auc_results"
)

print(latex_code)

\begin{table}
\caption{AUC of Loss across $\epsilon$ values with Relative Improvements}
\label{tab:auc_results}
\begin{tabular}{lrrr|r|rrrrrr|r}
\toprule
Dataset & $N$ & $d$ & $k$ & Non-Priv & PE & HDPE & Google & FastLloyd & DP-Lib & ICML17 & Improv. \\
\midrule
birch2 & 25000 & 2 & 100 & 0.0001 & \bfseries {\cellcolor[HTML]{D4EDDA}} 0.0003 & - & 0.0658 & 0.0031 & 0.0124 & 0.0161 & +90.92\% \\
iris & 150 & 4 & 3 & 0.1361 & \bfseries {\cellcolor[HTML]{D4EDDA}} 0.2894 & - & 0.6892 & 0.3979 & 1.3048 & 5.5766 & +27.28\% \\
adult & 48842 & 6 & 3 & 0.0056 & \bfseries {\cellcolor[HTML]{D4EDDA}} 0.0056 & - & 0.0076 & 0.0155 & 0.0213 & 0.0113 & +25.55\% \\
mnist & 70000 & 84 & 10 & 0.1958 & 0.2920 & \bfseries {\cellcolor[HTML]{D4EDDA}} 0.2169 & 0.2173 & 0.5822 & 0.5315 & 0.7904 & +0.16\% \\
letter & 20000 & 16 & 26 & 0.2481 & \bfseries {\cellcolor[HTML]{D4EDDA}} 0.2852 & - & 0.3187 & 0.3963 & 0.4341 & 0.6277 & +10.50\% \\
gas & 36733 & 12 & 6 & 0.1026 & \bfseries {\cellcolor[HTML]{D4EDDA}} 0.1

In [6]:
avg_improvement = df['Improv.'].mean()
print(f"Average improvement across all datasets: {avg_improvement:+.2f}%")

Average improvement across all datasets: +26.14%
